# 8.2 Integración LangChain + Ollama

Integra búsqueda semántica (Qdrant + Neo4j) con **Ollama** para respuestas en lenguaje natural.

## Requisitos:
1. ✅ Ollama instalado y corriendo
2. 📥 El modelo se descargará automáticamente (llama3)
3. 🔗 **Servidor FastAPI debe estar corriendo** (ejecuta notebook 8.1 primero)

## Pasos:
1. Ejecuta Cell 1: Instalar dependencias
2. Ejecuta Cell 2: Importar librerías
3. Ejecuta Cell 3: Verificar Ollama
4. Ejecuta Cell 4: Descargar modelo llama3 (varios minutos)
5. Ejecuta Cell 5: Crear retriever
6. Ejecuta Cell 6-8: Probar RAG

In [1]:
!pip install -q langchain langchain-community langchain-core langchain-ollama requests

In [2]:
from langchain_ollama import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_core.runnables import RunnablePassthrough
import requests
import json

# También para verificación de servicios
from qdrant_client import QdrantClient
from neo4j import GraphDatabase

print('✅ Librerías importadas')

✅ Librerías importadas


In [3]:
# Verificar Ollama
try:
    resp = requests.get('http://localhost:11434/api/tags', timeout=2)
    models = [m['name'] for m in resp.json().get('models', [])]
    print(f'✅ Ollama corriendo')
    print(f'📦 Modelos: {', '.join(models[:3])}')
except:
    print('❌ Ollama no está corriendo')
    print('💡 Instalar: https://ollama.ai')
    print('💡 Luego: ollama pull llama3')

✅ Ollama corriendo
📦 Modelos: llama3:latest


In [4]:
# Descargar modelo si no existe
MODEL = 'llama3'
print(f'📥 Verificando si el modelo {MODEL} está disponible...')

try:
    models = requests.get('http://localhost:11434/api/tags', timeout=2).json()
    model_names = [m['name'].split(':')[0] for m in models.get('models', [])]
    
    if MODEL in model_names:
        print(f'✅ Modelo {MODEL} ya está instalado')
    else:
        print(f'📦 Descargando modelo {MODEL} (esto puede tardar varios minutos)...')
        response = requests.post(f'http://localhost:11434/api/pull', 
                              json={'name': MODEL}, 
                              stream=True)
        
        for line in response.iter_lines():
            if line:
                try:
                    data = json.loads(line)
                    if 'status' in data:
                        print(f"   {data['status']}")
                    if 'completed' in data and data['completed'] == 100:
                        print(f'✅ Modelo {MODEL} descargado exitosamente')
                        break
                except:
                    continue
        
except Exception as e:
    print(f'⚠️ Error: {e}')
    print(f'💡 Intenta descargar manualmente desde: https://ollama.ai')
    print(f'   O ejecuta: ollama pull {MODEL}')
    
llm = OllamaLLM(model=MODEL, base_url='http://localhost:11434')
print(f'✅ LLM {MODEL} cargado')

📥 Verificando si el modelo llama3 está disponible...
✅ Modelo llama3 ya está instalado
✅ LLM llama3 cargado


In [5]:
# Verificar que FastAPI esté corriendo
print("="*70)
print("VERIFICANDO SERVICIOS REQUERIDOS")
print("="*70)

# Verificar FastAPI
try:
    resp = requests.get('http://127.0.0.1:8000/health', timeout=2)
    if resp.status_code == 200:
        print("✅ FastAPI está corriendo")
    else:
        print("⚠️ FastAPI respondió con código:", resp.status_code)
except:
    print("❌ FastAPI no está corriendo")
    print("💡 Ejecuta el notebook 8.1 primero para iniciar el servidor")

# Verificar Qdrant
try:
    from qdrant_client import QdrantClient
    client = QdrantClient(url="http://localhost:6333")
    collections = client.get_collections().collections
    print(f"✅ Qdrant está corriendo ({len(collections)} colecciones)")
except:
    print("❌ Qdrant no está corriendo")

# Verificar Neo4j
try:
    from neo4j import GraphDatabase
    driver = GraphDatabase.driver("neo4j://127.0.0.1:7687", auth=("neo4j", "proyectotec"))
    with driver.session() as session:
        result = session.run("RETURN 1 AS test")
        print("✅ Neo4j está corriendo")
    driver.close()
except:
    print("⚠️ Neo4j no pudo conectarse (verifica usuario/password)")

print("\n" + "="*70)


VERIFICANDO SERVICIOS REQUERIDOS
✅ FastAPI está corriendo
✅ Qdrant está corriendo (3 colecciones)
✅ Neo4j está corriendo



In [6]:
class AskRetriever:
    def __init__(self, type_in='recall'):
        self.type_in = type_in
        # Definir mapas de regiones a marcas
        self.region_brands = {
            'european': ['BMW', 'Mercedes-Benz', 'Audi', 'Volkswagen', 'VW', 'Porsche', 
                        'Jaguar', 'Land Rover', 'Range Rover', 'Volvo', 'Mini', 'Fiat',
                        'Peugeot', 'Renault', 'Citroen', 'Alfa Romeo', 'Maserati',
                        'Ferrari', 'Lamborghini', 'Bentley', 'Rolls-Royce', 'Aston Martin',
                        'Saab', 'Opel', 'Seat', 'Skoda', 'Alpine'],
            'american': ['Ford', 'Chevrolet', 'Chevy', 'GMC', 'Cadillac', 'Buick',
                       'Chrysler', 'Dodge', 'Jeep', 'Ram', 'Lincoln', 'Tesla'],
            'asian': ['Toyota', 'Honda', 'Nissan', 'Mazda', 'Subaru', 'Mitsubishi',
                     'Lexus', 'Acura', 'Infiniti', 'Hyundai', 'Kia', 'Genesis',
                     'Suzuki', 'Isuzu'],
            'chinese': ['BYD', 'Geely', 'Chery', 'Great Wall', 'SAIC', 'MG', 'NIO']
        }
    
    def expand_geographic_query(self, query: str) -> str:
        """Expande consultas geográficas a marcas específicas"""
        query_lower = query.lower()
        
        # Detectar región
        region = None
        if any(term in query_lower for term in ['europe', 'european', 'europeo', 'alemán', 'alemania']):
            region = 'european'
        elif any(term in query_lower for term in ['american', 'americano', 'usa', 'us']):
            region = 'american'
        elif any(term in query_lower for term in ['asian', 'asiatico', 'asiático']):
            region = 'asian'
        elif any(term in query_lower for term in ['chinese', 'chino', 'china']):
            region = 'chinese'
        
        if region and region in self.region_brands:
            brands = ' '.join(self.region_brands[region])
            expanded_query = f"{query} {brands}"
            return expanded_query
        
        return query
    
    def get_relevant_documents(self, query: str):
        # Expandir query si es geográfica
        expanded_query = self.expand_geographic_query(query)
        if expanded_query != query:
            print(f"🔍 Query expandido: {expanded_query[:100]}")
        
        # Usar API de FastAPI en lugar de función local
        try:
            # FastAPI espera 'question', no 'query'
            resp = requests.post('http://127.0.0.1:8000/search', 
                                json={'question': expanded_query, 'k': 10, 'type_in': self.type_in},
                                timeout=10)
            
            if resp.status_code == 200:
                results = resp.json()['results']
            else:
                print(f'⚠️ Error API: {resp.status_code}')
                results = []
            
            docs = []
            for r in results:
                # Formatear contenido del documento
                content_parts = []
                
                # ID
                if 'id' in r:
                    content_parts.append(f'[ID: {r["id"]}]')
                
                # Make, Model, Year
                if all(k in r for k in ['make', 'model', 'year']):
                    content_parts.append(f'{r["make"]} {r["model"]} ({r["year"]})')
                
                # Component
                if 'component' in r:
                    content_parts.append(f'\nComponente: {r["component"]}')
                
                # Text
                if 'text' in r:
                    text = r['text'][:500] if len(r['text']) > 500 else r['text']
                    content_parts.append(f'\n{text}')
                elif 'description' in r:
                    desc = r['description'][:500] if len(r['description']) > 500 else r['description']
                    content_parts.append(f'\n{desc}')
                
                # Consequence o subject
                if 'consequence' in r:
                    content_parts.append(f'\nConsecuencia: {r["consequence"]}')
                if 'subject' in r:
                    content_parts.append(f'\nAsunto: {r["subject"]}')
                
                content = '\n'.join(content_parts)
                docs.append(Document(page_content=content, metadata={'id': r.get('id', ''), 'type': self.type_in}))
            
            return docs
            
        except requests.exceptions.RequestException as e:
            print(f'❌ Error conectando con API: {e}')
            print('💡 Asegúrate de que el servidor FastAPI esté corriendo (notebook 8.1)')
            return []

print('✅ Retriever creado')

✅ Retriever creado


In [7]:
# Test
query = 'airbag sensor failure'
print(f'Pregunta: {query}\n')

retriever = AskRetriever()
docs = retriever.get_relevant_documents(query)
print(f'Encontrados: {len(docs)} documentos')
print('\nPrimer documento:')
print(docs[0].page_content[:200] if docs else 'Sin resultados')

Pregunta: airbag sensor failure

Encontrados: 10 documentos

Primer documento:
[ID: 22V240000]
BMW IX (2023)

Componente: AIR BAGS: AIR BAG/RESTRAINT CONTROL MODULE:SOFTWARE

BMW of North America, LLC (BMW) is recalling certain 2022-2023 iX xDrive40, iX XDrive50, and iX M60 vehi


In [9]:
# Configurar retriever
retriever = AskRetriever(type_in='recall')
print('✅ Retriever configurado para RECALLS')

# Crear RAG chain
template = '''Eres asistente de NHTSA (National Highway Traffic Safety Administration). 
Basándote únicamente en el siguiente contexto de seguridad vehicular:

{context}

INSTRUCCIONES:
1. Si la pregunta menciona regiones (European, American, Asian, Chinese), agrupa las marcas por región:
   - European makers: BMW, Mercedes-Benz, Audi, Volkswagen, VW, Porsche, Jaguar, Land Rover, Volvo, Mini, Fiat, Peugeot, Renault, etc.
   - American makers: Ford, Chevrolet, GMC, Cadillac, Chrysler, Dodge, Jeep, Ram, Lincoln, Tesla, etc.
   - Asian makers: Toyota, Honda, Nissan, Mazda, Hyundai, Kia, etc.
   - Chinese makers: BYD, Geely, Chery, etc.

2. Analiza el contexto y cuenta/describe las marcas relevantes según la región preguntada.

3. Responde de forma concisa y clara.

4. Si no hay suficientes marcas en el contexto para responder, así lo indicarás.

Pregunta: {question}
Respuesta:'''

prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    return '\n\n---\n\n'.join([f'Documento {i+1}:\n{d.page_content}' for i, d in enumerate(docs)])

# Crear chain
chain = (
    {"context": lambda x: format_docs(retriever.get_relevant_documents(x["question"])),
     "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print('✅ RAG Chain listo')

✅ Retriever configurado para RECALLS
✅ RAG Chain listo


In [10]:
# Test completo
query = 'What are common airbag failures?'
print('Pregunta:', query)
print('='*50)

try:
    ans = chain.invoke({"question": query})
    print(ans)
except Exception as e:
    print('Error:', e)

Pregunta: What are common airbag failures?
Based on the provided documents, common airbag failures mentioned include:

* Inflator rupture due to propellant degradation occurring after long-term exposure to absolute humidity and temperature cycling.
* Passenger side frontal air bag inflators may fracture when deployed, allowing gas to vent behind the instrument panel instead of into the air bag.
* Air bag doors may detach during deployment, increasing the risk of injury.
* Passenger air bags may not fully inflate due to welding issues on the inflator.
* Inflators may tear or not properly unfold during deployment, increasing the risk of injury.

Note that these failures are specific to certain makes and models, as indicated in the recall notices.


In [29]:
# Test completo
query = 'Which makers have more recalls? Chinese, European or American makers?'
print('Pregunta:', query)
print('='*50)

try:
    ans = chain.invoke({"question": query})
    print(ans)
except Exception as e:
    print('Error:', e)

Pregunta: Which makers have more recalls? Chinese, European or American makers?
Based on the provided context, I can see that there are recalls from International Motors, LLC (International), Orange EV (OEV), Kia America, Inc. (Kia), Autocar, LLC (Autocar), Chrysler, and Braun Corporation. All of these companies are American makers.

Therefore, it appears that American makers have more recalls in this context.


In [33]:
# Test completo
query = 'What information do you have about the recalls of European makers?'
print('Pregunta:', query)
print('='*50)

try:
    ans = chain.invoke({"question": query})
    print(ans)
except Exception as e:
    print('Error:', e)

Pregunta: What information do you have about the recalls of European makers?
🔍 Query expandido: What information do you have about the recalls of European makers? BMW Mercedes-Benz Audi Volkswagen
Based on the provided documents, I can identify several European makes affected by recalls:

* BMW (Documents 1, 6)
* Mercedes-Benz (Document 10)
* Audi (Document 2)
* Volkswagen (Documents 3, 4, 9)

These recalls affect various models and components, including seat belts, fuel systems, steering gearboxes, air bags, and more.
